# 05 - Class 1 Reserved Slot Strategy

This notebook compares three booking rules:

- pooled FCFS baseline with no protected slots
- strict Class 1 reservation: 10 of 32 daily slots are protected for Class 1 only
- released Class 1 reservation: Class 1 gets first pass on the 10 protected slots, then Class 2 may fill unused protected capacity

The comparison uses the same baseline demand, balking, cancellation, no-show, horizon, and measurement settings. Only the booking policy changes.

In [ ]:
from __future__ import annotations

from dataclasses import replace
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(iterable=None, **kwargs):
        return iterable


def find_repo_dir(start: Path) -> Path:
    current = Path(start).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "configs" / "baseline.yaml").exists() and (
            candidate / "simulation" / "config_loader.py"
        ).exists():
            return candidate
    raise FileNotFoundError("Could not find the repository root from the current notebook location.")


REPO_DIR = find_repo_dir(Path.cwd())
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from analysis.metrics import aggregate_result_row, class_result_rows
from simulation.config_loader import load_config
from simulation.engine import ClinicAppointmentSimulation

plt.style.use("default")

## Policy Setup

The protected pool is 10 slots per appointment day. The remaining 22 slots are the general pool.

In the strict policy, Class 2 can never book the protected pool. In the released policy, daily Class 1 arrivals get first pass on protected slots; after that, Class 2 can use protected slots that Class 1 did not take.

In [ ]:
BASE_CONFIG = load_config(REPO_DIR / "configs" / "baseline.yaml")
RESERVED_CLASS_ID = 1
RESERVED_SLOTS = 10

policy_configs = {
    "Pooled FCFS": BASE_CONFIG,
    "Strict C1 reservation": replace(
        BASE_CONFIG,
        reserved_class_id=RESERVED_CLASS_ID,
        reserved_slots_per_day=RESERVED_SLOTS,
        release_reserved_slots=False,
    ),
    "Released C1 reservation": replace(
        BASE_CONFIG,
        reserved_class_id=RESERVED_CLASS_ID,
        reserved_slots_per_day=RESERVED_SLOTS,
        release_reserved_slots=True,
    ),
}

policy_table = pd.DataFrame(
    [
        {
            "policy": name,
            "reserved_class_id": cfg.reserved_class_id,
            "reserved_slots_per_day": cfg.reserved_slots_per_day,
            "general_slots_per_day": cfg.slots_per_day - cfg.reserved_slots_per_day,
            "release_reserved_slots": cfg.release_reserved_slots,
        }
        for name, cfg in policy_configs.items()
    ]
)

display(policy_table)

## Multi-Seed Runs

A single run can be noisy because daily arrivals, balking, cancellations, and no-shows are random. The next cell runs each policy over the same seed set and stores aggregate and class-level rows.

In [ ]:
SEEDS = list(range(5101, 5131))

aggregate_rows = []
class_rows = []

for policy, config in policy_configs.items():
    for seed in tqdm(SEEDS, desc=policy):
        result = ClinicAppointmentSimulation(replace(config, seed=seed)).run()
        fixed = {"policy": policy, "seed": seed}
        aggregate_rows.append(aggregate_result_row(result, fixed))
        class_rows.extend(class_result_rows(result, fixed))

aggregate_df = pd.DataFrame(aggregate_rows)
class_df = pd.DataFrame(class_rows)

for outcome in ["balked", "no_offer", "canceled", "no_show", "unresolved_booked"]:
    aggregate_df[f"{outcome}_rate"] = (
        aggregate_df[f"total_{outcome}"] / aggregate_df["total_arrivals"]
    )

aggregate_df["served_rate"] = aggregate_df["overall_percent_serviced"]
aggregate_df.head()

## Aggregate Comparison

These metrics show the access-capacity tradeoff. Strict reservation should protect Class 1 capacity, but it can leave slots unused when Class 1 demand does not reach the protected pool. Released reservation tests whether Class 2 can recover that unused capacity.

In [ ]:
aggregate_metrics = [
    "average_utilization",
    "served_rate",
    "mean_offered_booking_delay",
    "mean_accepted_booking_delay",
    "balked_rate",
    "no_offer_rate",
    "canceled_rate",
    "no_show_rate",
]

aggregate_summary = (
    aggregate_df.groupby("policy")[aggregate_metrics]
    .agg(["mean", "std"])
    .reindex(policy_configs.keys())
    .round(3)
)

display(aggregate_summary)

In [ ]:
plot_metrics = [
    ("average_utilization", "Average utilization"),
    ("served_rate", "Served rate"),
    ("no_offer_rate", "No-offer rate"),
    ("mean_offered_booking_delay", "Mean offered delay"),
]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for ax, (metric, title) in zip(axes.flat, plot_metrics):
    means = aggregate_df.groupby("policy")[metric].mean().reindex(policy_configs.keys())
    stds = aggregate_df.groupby("policy")[metric].std().reindex(policy_configs.keys())
    means.plot(kind="bar", yerr=stds, capsize=3, ax=ax, color=["#6b7280", "#2563eb", "#059669"])
    ax.set_title(title)
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=25)
    ax.grid(axis="y", alpha=0.25)

fig.tight_layout()

## Class-Level Effects

The reservation policy is designed around Class 1, so class-level served rate, delay, and no-offer rate are the main diagnostics. A policy can look good in aggregate while shifting access between classes.

In [ ]:
class_df["no_offer_rate"] = (class_df["no_offer"] / class_df["arrivals"]).fillna(0)
class_df["balking_rate"] = (class_df["balked"] / class_df["offered"]).fillna(0)

class_metrics = [
    "percent_serviced",
    "mean_offered_booking_delay",
    "no_offer_rate",
    "balking_rate",
]

class_summary = (
    class_df.groupby(["policy", "class_id"])[class_metrics]
    .mean()
    .reindex(policy_configs.keys(), level="policy")
    .round(3)
)

display(class_summary)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

served_by_class = (
    class_df.groupby(["policy", "class_id"])["percent_serviced"]
    .mean()
    .unstack("class_id")
    .reindex(policy_configs.keys())
)
delay_by_class = (
    class_df.groupby(["policy", "class_id"])["mean_offered_booking_delay"]
    .mean()
    .unstack("class_id")
    .reindex(policy_configs.keys())
)

served_by_class.plot(kind="bar", ax=axes[0], color=["#2563eb", "#dc2626"])
axes[0].set_title("Served rate by class")
axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=25)
axes[0].grid(axis="y", alpha=0.25)

delay_by_class.plot(kind="bar", ax=axes[1], color=["#2563eb", "#dc2626"])
axes[1].set_title("Mean offered delay by class")
axes[1].set_xlabel("")
axes[1].tick_params(axis="x", rotation=25)
axes[1].grid(axis="y", alpha=0.25)

fig.tight_layout()

## Reading The Result

Use the strict policy to measure the cost of true protection: Class 2 cannot touch the 10 reserved slots, so unused protected capacity can show up as lower utilization or higher Class 2 no-offer.

Use the released policy to measure the recovery value of flexible protection: Class 1 still gets first pass on the protected pool, but Class 2 can backfill unused capacity. That should usually improve utilization and reduce no-offer, while weakening the guarantee that future Class 1 arrivals will still find protected capacity.